# EMA 9/21 Crypto Trading Agent — Colab GPU Training

Trains the GRU trend model for every (symbol, timeframe) pair in `config.yaml`,
backtests the EMA 9/21 crossover to rank the **best timescales**, and exports the
trained model bundles (`models/*.pt`) for the predictor (`src/predict.py`).

**Before running:** `Runtime → Change runtime type → GPU (T4 is fine)`.

> ⚠️ Educational tooling, not financial advice. Backtest results do not guarantee future performance — paper trade before risking funds.

In [ ]:
# 1) Verify the GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU detected — switch runtime type to GPU for much faster training')

In [ ]:
# 2) Clone the repo and install dependencies (torch is pre-installed on Colab)
REPO_URL = 'https://github.com/sakethsurabhi967-cmd/crypto-strategies-trianer.git'
BRANCH   = 'claude/crypto-trading-ema-agent-h4hi0j'  # change to 'main' after merging

import os
if not os.path.exists('crypto-strategies-trianer'):
    !git clone --branch {BRANCH} {REPO_URL}
%cd crypto-strategies-trianer
!pip -q install ccxt pyyaml

In [ ]:
# 3) (Optional) tweak what gets trained without editing config.yaml
SYMBOLS    = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']
TIMEFRAMES = ['15m', '30m', '1h', '4h']
MARKET     = 'spot'      # 'spot' or 'futures' (futures adds short signals)

sym_args = ' '.join(SYMBOLS)
tf_args  = ' '.join(TIMEFRAMES)
print(f'Will train: {SYMBOLS} x {TIMEFRAMES} on {MARKET}')

In [ ]:
# 4) Train on GPU — one model bundle per (symbol, timeframe)
!python -m src.train --symbols {sym_args} --timeframes {tf_args} --market {MARKET}

import glob
trained = glob.glob('models/*.pt')
assert trained, (
    'No model bundles were produced — scroll up in this cell for FAILED lines. '
    'The usual cause is a data-download problem; the code already falls back to '
    'several exchanges and the Binance public data mirror, so check the exact error.'
)
print(f'\n{len(trained)} model bundle(s): {sorted(trained)}')

In [ ]:
# 5) (Optional) also train futures models so the predictor can go SHORT
TRAIN_FUTURES = False
if TRAIN_FUTURES:
    !python -m src.train --symbols {sym_args} --timeframes {tf_args} --market futures

In [ ]:
# 6) Backtest & rank timescales — raw EMA crossover vs model-filtered
!python -m src.backtest --symbols {sym_args} --timeframes {tf_args} --market {MARKET}
!python -m src.backtest --symbols {sym_args} --timeframes {tf_args} --market {MARKET} --use-model

In [ ]:
# 7) Inspect the ranking — pick your best timescale per symbol
import json, os, pandas as pd
assert os.path.exists('models/backtest_ranking.json'), (
    'models/backtest_ranking.json is missing — the backtest in cell 6 did not finish. '
    'Scroll through cell 6 output for the actual error (look for FAILED / Traceback), '
    'then re-run cell 6. If cell 4 produced no models, fix that first.'
)
with open('models/backtest_ranking.json') as f:
    ranking = json.load(f)
df = pd.DataFrame(ranking['ranking']).sort_values(['symbol','sharpe'], ascending=[True, False])
display(df)
print('\nBest per symbol:')
display(pd.DataFrame(ranking['best_per_symbol']))

In [ ]:
# 8) Export the trained models for the predictor
!zip -qr trained_models.zip models/
print('trained_models.zip ready')

# 8a) direct download
from google.colab import files
files.download('trained_models.zip')

In [ ]:
# 8b) (Optional) save to Google Drive instead
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp trained_models.zip /content/drive/MyDrive/
    print('Copied to Google Drive: MyDrive/trained_models.zip')

## Using the trained models locally

On your own machine:

```bash
git clone https://github.com/sakethsurabhi967-cmd/crypto-strategies-trianer.git
cd crypto-strategies-trianer
pip install -r requirements.txt
unzip trained_models.zip          # puts the .pt bundles into models/
python -m src.predict --loop 60   # live signals every 60 s
```

The predictor prints one JSON signal per model, e.g.
`{"symbol": "BTC/USDT", "timeframe": "1h", "ema_state": "bullish", "prob_up": 0.63, "action": "LONG", ...}` — feed it into your own execution/alerting layer.